# 04 — Spatial Probability Map

Takes a single paired tile from the val set and slides a sub-window
across it to produce a **continuous spatial bleaching probability map**
at sub-tile resolution.

Works because ResNet's global average pooling accepts any spatial input
size — 48×48 sub-patches can be scored even though the model was trained
on 120×120 tiles.

Output: three panels — RGB | probability heatmap | heatmap overlaid on RGB.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CKPT_PATH   = "/pscratch/sd/k/kevinval/coraltest/ct_classifier/model_states/best.pt"
PROJECT_DIR = "/pscratch/sd/k/kevinval/coraltest/ct_classifier"
FIGURES_DIR = "figures"

SPLIT        = "val"     # which split to sample from
SEED         = None      # None = random tile every run; int for reproducibility

# Sliding-window settings (applied within the 120×120 tile)
WINDOW_SIZE  = 48        # sub-patch size fed to the model
STRIDE       = 4         # pixel step between windows
BATCH_SIZE   = 128       # windows per forward pass

# RGB display
RGB_GAMMA    = 1.35      # >1 darkens midtones
RGB_GAIN     = 0.95

# Overlay
HEATMAP_CMAP      = "turbo"
OVERLAY_ALPHA_MAX = 0.30   # max opacity of heatmap over RGB
ALPHA_THRESHOLD   = 0.18   # below this prob, heatmap is invisible
ALPHA_POWER       = 1.6

In [ ]:
import os, sys, random, time
import yaml
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from ct_classifier.dataset import BleachDataset
from ct_classifier.model import CustomResNet

os.makedirs(FIGURES_DIR, exist_ok=True)

cfg_path = os.path.join(os.path.dirname(CKPT_PATH), "config.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = CustomResNet(cfg["num_classes"], cfg["layers"])
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state["model"])
model.to(device).eval()

# Seed
seed = int(time.time() * 1000) % (2**32 - 1) if SEED is None else int(SEED)
print(f"Seed: {seed}")
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
g = torch.Generator(); g.manual_seed(seed)

dl = DataLoader(
    BleachDataset(cfg, split=SPLIT, eval=True),
    batch_size=1, shuffle=True, generator=g,
    num_workers=cfg.get("num_workers", 0),
)

In [ ]:
# ── Sample one tile ───────────────────────────────────────────────────────────
data, labels, (id1, id2) = next(iter(dl))
x = data[0].cpu()       # [18, H, W]
y = int(labels[0].item())
C, H, W = x.shape
class_names = ["healthy", "bleached"]

# Whole-tile prediction
with torch.no_grad():
    p_full = torch.softmax(model(data.to(device)), dim=1)[0].cpu().numpy()
pred_full = int(np.argmax(p_full))

print(f"Tile shape : {(C, H, W)}")
print(f"True label : {class_names[y]}")
print(f"Prediction : {class_names[pred_full]}  P(bleached)={p_full[1]:.3f}")

In [ ]:
# ── Sliding-window inference ──────────────────────────────────────────────────
def build_windows(H, W, win, stride):
    ys = list(range(0, H - win + 1, stride))
    xs = list(range(0, W - win + 1, stride))
    if not ys or ys[-1] != H - win: ys.append(H - win)
    if not xs or xs[-1] != W - win: xs.append(W - win)
    return [(y, x) for y in ys for x in xs]

def gaussian_kernel(size, sigma_frac=0.20):
    ax = np.arange(size) - (size - 1) / 2.0
    xx, yy = np.meshgrid(ax, ax)
    sigma = sigma_frac * size
    w = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return (w / w.max()).astype(np.float32)

coords  = build_windows(H, W, WINDOW_SIZE, STRIDE)
kernel  = gaussian_kernel(WINDOW_SIZE)
prob_sum   = np.zeros((H, W), dtype=np.float32)
weight_sum = np.zeros((H, W), dtype=np.float32)

patches, metas = [], []

with torch.no_grad():
    for idx, (y0, x0) in enumerate(coords):
        patches.append(x[:, y0:y0+WINDOW_SIZE, x0:x0+WINDOW_SIZE])
        metas.append((y0, x0))

        if len(patches) == BATCH_SIZE or idx == len(coords) - 1:
            batch_p = torch.stack(patches).to(device)
            probs   = torch.softmax(model(batch_p), dim=1)[:, 1].cpu().numpy()
            for p, (y0, x0) in zip(probs, metas):
                prob_sum  [y0:y0+WINDOW_SIZE, x0:x0+WINDOW_SIZE] += p * kernel
                weight_sum[y0:y0+WINDOW_SIZE, x0:x0+WINDOW_SIZE] += kernel
            patches, metas = [], []

prob_map = np.clip(prob_sum / (weight_sum + 1e-8), 0, 1)
print(f"Done. {len(coords)} windows. Prob range: {prob_map.min():.3f}–{prob_map.max():.3f}")

In [ ]:
# ── Display ───────────────────────────────────────────────────────────────────
from utils.viz import to_display_rgb, percentile_stretch

# RGB from query image (channels 8–15)
query_bands = x[8:16]    # [8, H, W]
rgb = percentile_stretch(to_display_rgb(query_bands))
rgb = np.clip(rgb ** RGB_GAMMA * RGB_GAIN, 0, 1)

# valid pixel mask (at least one channel nonzero)
valid = np.any(np.abs(x[:8].numpy()) > 1e-12, axis=0)

# heatmap overlay with alpha proportional to probability
cmap = matplotlib.colormaps[HEATMAP_CMAP]
norm = Normalize(vmin=0, vmax=1)
heat_rgb = cmap(norm(prob_map))[..., :3].astype(np.float32)

alpha = np.clip(
    (prob_map - ALPHA_THRESHOLD) / (1.0 - ALPHA_THRESHOLD + 1e-8), 0, 1
) ** ALPHA_POWER * OVERLAY_ALPHA_MAX
alpha[~valid] = 0.0

overlay = np.clip((1 - alpha[..., None]) * rgb + alpha[..., None] * heat_rgb, 0, 1)
overlay[~valid] = 0.0

prob_show = np.where(valid, prob_map, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

axes[0].imshow(rgb)
axes[0].set_title("Query tile (RGB)"); axes[0].axis("off")

im = axes[1].imshow(prob_show, cmap=HEATMAP_CMAP, vmin=0, vmax=1)
axes[1].set_title("Bleaching probability"); axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title("Overlay"); axes[2].axis("off")

fig.colorbar(
    ScalarMappable(norm=norm, cmap=cmap),
    ax=axes, fraction=0.025, pad=0.02,
    label="P(bleached)"
)
fig.suptitle(
    f"True={class_names[y]}  |  Pred={class_names[pred_full]}  "
    f"|  P(bleached)={p_full[1]:.3f}  |  ids=({int(id1[0])},{int(id2[0])})",
    fontsize=10
)
fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, f"spatial_prob_map_seed{seed}.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()